In [3]:
import requests
import datetime
from urllib.parse import urlparse

def get_domain_age(url):
    try:
        # 1. Extraire le domaine (ex: google.com)
        parsed_uri = urlparse(url)
        domain = parsed_uri.netloc
        if domain.startswith('www.'):
            domain = domain[4:]
        
        # 2. On interroge l'annuaire public RDAP (méthode moderne)
        response = requests.get(f"https://rdap.org/domain/{domain}", timeout=5)
        
        # 3. Si le serveur nous répond correctement
        if response.status_code == 200:
            data = response.json()
            for event in data.get('events', []):
                if event['eventAction'] == 'registration':
                    date_str = event['eventDate']
                    # On nettoie la date pour Python
                    date_str = date_str.replace('Z', '+00:00')
                    creation_date = datetime.datetime.fromisoformat(date_str)
                    
                    # On calcule l'âge
                    age = (datetime.datetime.now(datetime.timezone.utc) - creation_date).days
                    return age
        return -1 # Si pas de date trouvée
    except Exception as e:
        return -1 # Si erreur réseau

# --- TEST ---
urls_test = [
    "https://www.google.com",
    "https://www.wikipedia.org",
    "https://www.facebook.com",
    "http://paypal-login-verify.xyz",
    "http://secure-update-bank.com"
]

for u in urls_test:
    print(f"URL : {u} | Âge : {get_domain_age(u)} jours")

URL : https://www.google.com | Âge : 10657 jours
URL : https://www.wikipedia.org | Âge : 9441 jours
URL : https://www.facebook.com | Âge : 10827 jours
URL : http://paypal-login-verify.xyz | Âge : -1 jours
URL : http://secure-update-bank.com | Âge : -1 jours


In [ ]:
import pandas as pd
import sys
sys.path.append('../src')
from features import *

# 1. On recharge le fichier de base (avec le texte des URL)
df_base = pd.read_csv('../data/archive (2)/new_data_urls.csv')

# 2. On prend un échantillon de 10 000 URL (mélangées)
df_sample = df_base.sample(n=10000, random_state=42).copy()

# 3. On calcule les features de texte
df_sample['longueur'] = df_sample['url'].apply(url_length)
df_sample['nb_points'] = df_sample['url'].apply(count_dots)
df_sample['nb_tirets'] = df_sample['url'].apply(count_hyphens)
df_sample['nb_chiffres'] = df_sample['url'].apply(count_digits)
df_sample['nb_arobase'] = df_sample['url'].apply(count_at_symbols)
df_sample['a_ip'] = df_sample['url'].apply(has_ip_address)
df_sample['a_https'] = df_sample['url'].apply(has_https)
df_sample['nb_sous_domaines'] = df_sample['url'].apply(count_subdomains)
df_sample['mots_suspects'] = df_sample['url'].apply(has_suspicious_words)

print("Échantillon de 10000 URL prêt avec les features de texte !")

Échantillon de 1000 URL prêt avec les features de texte !


In [16]:
import requests
import datetime
from urllib.parse import urlparse
from tqdm import tqdm
import time

# On utilise la NOUVELLE fonction détaillée (celle qui donne les raisons)
def get_domain_age_detailed(url):
    try:
        #ajouter le http s'il n'est pas present 
        if not url.startswith('http://') and not url.startswith('https://'):
            url = 'http://' + url
            
        parsed_uri = urlparse(url)
        domain = parsed_uri.netloc
        if not domain:
            domain = parsed_uri.path.split('/')[0]
        
        if domain.startswith('www.'):
            domain = domain[4:]
        
        #appel au serveur central    
        response = requests.get(f"https://rdap.org/domain/{domain}", timeout=3)
        
        if response.status_code == 404: #dans le cas d'erreur 
            return -1 # Site fermé / inexistant
        if response.status_code == 429: # appel repétetif 
            return -3 # Bloqué par le serveur
            
        if response.status_code == 200: # appel avec sucesser
            data = response.json()
            for event in data.get('events', []):
                if event['eventAction'] == 'registration':
                    date_str = event['eventDate'].replace('Z', '+00:00')
                    creation_date = datetime.datetime.fromisoformat(date_str)
                    return (datetime.datetime.now(datetime.timezone.utc) - creation_date).days
            return -2 # Âge masqué
            
        return -1 
    except Exception:
        return -1

print("Calcul de l'âge des 10 000 domaines (avec analyse des erreurs)...")

# Le robot se met en marche !
df_sample['age_domaine'] = [get_domain_age_detailed(u) for u in tqdm(df_sample['url'])]

# LA SAUVEGARDE aDE SÉCURITÉ 
df_sample.to_csv('../data/sample_10000_reseau_v2.csv', index=False)

print("\nCalcul terminé et sauvegardé dans data/sample_10000_reseau_v2.csv !")

Calcul de l'âge des 10 000 domaines (avec analyse des erreurs)...


100%|██████████| 10000/10000 [1:02:58<00:00,  2.65it/s]



Calcul terminé et sauvegardé dans data/sample_10000_reseau_v2.csv !


In [17]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. LA CORRECTION : On attrape toutes les erreurs (-1, -2, -3)
df_sample['age_erreur'] = (df_sample['age_domaine'] < 0).astype(int)

# 2. Préparation des données
X = df_sample.drop(columns=['url', 'status'])
y = df_sample['status']

# 3. Séparation Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Entraînement de XGBoost
print("Entraînement du Modèle B optimisé...")
model_B = XGBClassifier(n_estimators=150, random_state=42, eval_metric='logloss', max_depth=5)
model_B.fit(X_train, y_train)

# 5. Évaluation
y_pred = model_B.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\nPrécision du Modèle B (Texte + Réseau optimisé) : {acc * 100:.2f} %")
print("\n--- Rapport détaillé ---")
print(classification_report(y_test, y_pred, target_names=["Phishing (0)", "Sain (1)"]))

# 6. Sauvegarde du modèle
joblib.dump(model_B, '../src/model_reseau.pkl')
print("\nNouveau Modèle B sauvegardé !")

Entraînement du Modèle B optimisé...

Précision du Modèle B (Texte + Réseau optimisé) : 80.60 %

--- Rapport détaillé ---
              precision    recall  f1-score   support

Phishing (0)       0.82      0.76      0.79       950
    Sain (1)       0.80      0.84      0.82      1050

    accuracy                           0.81      2000
   macro avg       0.81      0.80      0.80      2000
weighted avg       0.81      0.81      0.81      2000


Nouveau Modèle B sauvegardé !
